# Feature Engineering

Using the cleaned Boston 311 dataset (`master_311_clean.csv`) to explore features for a **Random Forest classification** model that predicts whether a neighborhood-month will be **high-risk** for needle pickup activity. The binary target directly supports the policy question: *which neighborhoods need proactive resource allocation?*

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import contextily as cx

sns.set_style("whitegrid")

df = pd.read_csv(Path(".") / "master_311_clean.csv", low_memory=False)

df["open_dt"] = pd.to_datetime(df["open_dt"], errors="coerce")
df["closed_dt"] = pd.to_datetime(df["closed_dt"], errors="coerce")

df["response_hours"] = (df["closed_dt"] - df["open_dt"]).dt.total_seconds() / 3600
df["month"] = df["open_dt"].dt.month
df["day_of_week"] = df["open_dt"].dt.dayofweek
df["is_needle"] = df["type"].str.contains("Needle Pickup", case=False, na=False).astype(int)

print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nFirst 3 rows:")
df.head(3)

ModuleNotFoundError: No module named 'contextily'

## Univariate Analysis

Distribution of each feature individually: histograms for numeric columns, bar charts for categorical columns.

In [ ]:
numeric_cols = ["latitude", "longitude", "year", "response_hours", "month", "day_of_week", "is_needle"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    data = df[col].dropna()
    if col == "response_hours":
        data = data.clip(upper=data.quantile(0.99))
    ax.hist(data, bins=40, edgecolor="black", alpha=0.7)
    ax.set_title(col)
    ax.set_ylabel("Count")

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Univariate Analysis — Numeric Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Response hours is so right skewed, it may help to remove the outlier.

In [ ]:
cat_cols = ["case_status", "subject", "reason", "type", "neighborhood",
            "neighborhood_services_district", "ward", "source"]

TOP_N = 15

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    counts = df[col].value_counts().head(TOP_N)
    counts.sort_values().plot.barh(ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(f"{col}  (top {min(TOP_N, len(counts))})")
    ax.set_xlabel("Count")

fig.suptitle("Univariate Analysis — Categorical Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Feature Engineering

### Response Hours — Remove Outliers, Log-Transform, and Neighborhood View

`response_hours` is heavily right-skewed with extreme outliers. We remove rows exceeding 700 hours as outliers, then log-transform the remaining values (`log1p` to handle zeros) to compress the tail. A bar chart of median response time by neighborhood shows whether certain areas consistently experience slower service.

In [ ]:
rows_before = len(df)
outliers_removed = (df["response_hours"] > 700).sum()

df = df[df["response_hours"].isna() | (df["response_hours"] <= 700)].copy()
df["response_hours_log"] = np.log1p(df["response_hours"].clip(lower=0))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df["response_hours"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_title("response_hours (after removing > 700h)")
axes[0].set_xlabel("Hours")
axes[0].set_ylabel("Count")

axes[1].hist(df["response_hours_log"].dropna(), bins=50, edgecolor="black", alpha=0.7, color="C2")
axes[1].set_title("After log1p transform")
axes[1].set_xlabel("log(1 + hours)")

axes[2].set_visible(False)

plt.suptitle("Response Hours — Before and After Transformation", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Rows before: {rows_before:,}")
print(f"Rows removed (> 700h): {outliers_removed:,}")
print(f"Rows after: {len(df):,}")
print(f"\nresponse_hours_log stats:\n{df['response_hours_log'].describe().to_string()}")

In [ ]:
median_resp = (df.dropna(subset=["response_hours"])
               .groupby("neighborhood")["response_hours"]
               .median()
               .sort_values(ascending=True))

fig, ax = plt.subplots(figsize=(10, 8))
median_resp.plot.barh(ax=ax, edgecolor="black", alpha=0.7, color="C3")
ax.set_title("Median Response Hours by Neighborhood (outliers > 700h removed)")
ax.set_xlabel("Median Hours")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### Multicollinearity — Categorical Features (Cramér's V)

For categorical variables, Pearson correlation does not apply. Instead we use ***Cramér's V***, which measures the strength of association between two categorical variables (0 = no association, 1 = perfect association). Values above ~0.7 suggest redundancy and a candidate for dropping one of the pair.

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    v = np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0.0
    return v

cat_cols = ["case_status", "subject", "reason", "type", "neighborhood",
            "neighborhood_services_district", "ward", "source"]

sample = df[cat_cols].sample(n=min(50_000, len(df)), random_state=42)

v_matrix = pd.DataFrame(np.zeros((len(cat_cols), len(cat_cols))),
                         index=cat_cols, columns=cat_cols)
for i, c1 in enumerate(cat_cols):
    for j, c2 in enumerate(cat_cols):
        if i <= j:
            val = cramers_v(sample[c1], sample[c2]) if i != j else 1.0
            v_matrix.loc[c1, c2] = val
            v_matrix.loc[c2, c1] = val

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(v_matrix.astype(float), annot=True, fmt=".2f", cmap="YlOrRd",
            square=True, linewidths=0.5, vmin=0, vmax=1, ax=ax)
ax.set_title("Cramér's V — Categorical Feature Association")
plt.tight_layout()
plt.show()

print("\nPairs with Cramér's V > 0.7 (high redundancy):")
for i, c1 in enumerate(cat_cols):
    for j, c2 in enumerate(cat_cols):
        if i < j and v_matrix.loc[c1, c2] > 0.7:
            print(f"  {c1}  ↔  {c2}  :  {v_matrix.loc[c1, c2]:.3f}")

### Temporal Features — Year, Month, and Quarter

We compare three temporal groupings (year, month, quarter) to see which captures the most meaningful variation in needle pickup rate. This informs which temporal feature(s) to include in the final model.

In [ ]:
df["quarter"] = df["open_dt"].dt.quarter

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Year
rate_year = df.groupby("year")["is_needle"].mean() * 100
axes[0].bar(rate_year.index.astype(str), rate_year.values, edgecolor="black", alpha=0.7)
axes[0].set_title("Needle Rate (%) by Year")
axes[0].set_ylabel("% of requests")
axes[0].tick_params(axis="x", rotation=45)

# Month
rate_month = df.groupby("month")["is_needle"].mean() * 100
axes[1].bar(rate_month.index.astype(str), rate_month.values, edgecolor="black", alpha=0.7, color="C1")
axes[1].set_title("Needle Rate (%) by Month")
axes[1].set_ylabel("% of requests")

# Quarter
rate_qtr = df.groupby("quarter")["is_needle"].mean() * 100
axes[2].bar(rate_qtr.index.astype(str), rate_qtr.values, edgecolor="black", alpha=0.7, color="C2")
axes[2].set_title("Needle Rate (%) by Quarter")
axes[2].set_ylabel("% of requests")

plt.suptitle("Temporal Grouping Comparison — Needle Rate", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

spread = {
    "Year":    rate_year.max() - rate_year.min(),
    "Month":   rate_month.max() - rate_month.min(),
    "Quarter": rate_qtr.max() - rate_qtr.min(),
}
print("Range of needle rate (%) across levels of each grouping:")
for k, v in spread.items():
    print(f"  {k:>8s}: {v:.2f} pp")

In [ ]:
nbhd_year = (df.groupby(["neighborhood", "year"])
             .agg(needle_count=("is_needle", "sum"),
                  total_requests=("is_needle", "size"))
             .reset_index())
nbhd_year["needle_share"] = nbhd_year["needle_count"] / nbhd_year["total_requests"]

nbhd_year = nbhd_year.sort_values(["neighborhood", "year"])
nbhd_year["needle_count_lag1"] = (nbhd_year.groupby("neighborhood")["needle_count"]
                                   .shift(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

valid = nbhd_year.dropna(subset=["needle_count_lag1"])

axes[0].scatter(valid["needle_count_lag1"], valid["needle_count"], alpha=0.5, s=30)
axes[0].set_xlabel("Needle Count (prior year)")
axes[0].set_ylabel("Needle Count (current year)")
axes[0].set_title("Lag-1 Feature vs Current Needle Count")
r = valid["needle_count_lag1"].corr(valid["needle_count"])
axes[0].annotate(f"r = {r:.2f}", xy=(0.05, 0.92), xycoords="axes fraction", fontsize=12)

top_nbhds = nbhd_year.groupby("neighborhood")["needle_count"].sum().nlargest(6).index
for nbhd in top_nbhds:
    sub = nbhd_year[nbhd_year["neighborhood"] == nbhd]
    axes[1].plot(sub["year"], sub["needle_count"], marker="o", label=nbhd)
axes[1].set_title("Needle Count Over Time — Top 6 Neighborhoods")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Needle Count")
axes[1].legend(fontsize=8, loc="upper left")

plt.suptitle("Lag Feature & Neighborhood Trends", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nCorrelation between lag-1 and current needle count: {r:.3f}")
print(f"\nAggregated neighborhood-year table ({len(nbhd_year)} rows):")
nbhd_year.head(10)

### Building the Final Feature Set

We aggregate request-level data to **neighborhood–month** rows and engineer the features the classifier will see. The target is a **binary label** — `high_risk` — indicating whether the needle pickup count for that neighborhood-month exceeds the **75th percentile** across all neighborhood-months.

This aligns directly with the policy question: resource allocation is a discrete decision (deploy crews or not), so a classifier that separates high-risk from low-risk periods is more actionable than a point-count prediction.

**Key decisions:**
- **Drop `type`:** It was used to derive `is_needle` (the target). Including it would leak the answer. `subject` and `reason` describe the department and category of each request — they are safe to include as proportion features per neighborhood-month.
- **Include `source`, `subject`, and `reason` as proportion features:** Each is broken into proportions of key categories per neighborhood-month (e.g., `source_app_pct`, `reason_street_cleaning_pct`). These capture the *mix* of 311 activity in a neighborhood, which signals neighborhood conditions.
- **Exclude neighborhood as a predictor:** Including neighborhood identity would let the model learn "Roxbury is high-risk" without understanding *why*. Dropping it forces the model to predict from observable conditions — producing results that generalize and are interpretable for policy makers. Neighborhood is still used for aggregation and the final risk map.
- **Response-time split features:** `pct_fast_close` (proportion closed within 3 hours) and `median_response_hours_log` capture the bimodal response-time pattern.
- **Temporal feature:** Only `month` is included to capture seasonality (summer peaks, winter dips). No lag or rolling-average features are used, since they are derived from the target variable.

In [ ]:
FAST_CLOSE_THRESHOLD = 3

print("Top source values:")
print(df["source"].value_counts().to_string())
print("\nTop subject values:")
print(df["subject"].value_counts().head(10).to_string())
print("\nTop reason values:")
print(df["reason"].value_counts().head(10).to_string())

agg = (df.groupby(["neighborhood", "year", "month"])
       .agg(
           needle_count=("is_needle", "sum"),
           total_requests=("is_needle", "size"),
           median_response_hours_log=("response_hours_log", "median"),
           pct_fast_close=("response_hours", lambda x: (x <= FAST_CLOSE_THRESHOLD).mean()),
           source_app_pct=("source", lambda x: (x == "Citizens Connect App").mean()),
           source_call_pct=("source", lambda x: (x == "Constituent Call").mean()),
           source_self_pct=("source", lambda x: (x == "Self Service").mean()),
           subject_pwd_pct=("subject", lambda x: (x == "Public Works Department").mean()),
           subject_bwsc_pct=("subject", lambda x: x.str.contains("Boston Water", case=False, na=False).mean()),
           subject_inspections_pct=("subject", lambda x: x.str.contains("Inspectional Services", case=False, na=False).mean()),
           reason_street_cleaning_pct=("reason", lambda x: (x == "Street Cleaning").mean()),
           reason_enforcement_pct=("reason", lambda x: (x == "Enforcement & Abandoned Vehicles").mean()),
           reason_sanitation_pct=("reason", lambda x: x.str.contains("Sanitation", case=False, na=False).mean()),
           reason_housing_pct=("reason", lambda x: x.str.contains("Housing", case=False, na=False).mean()),
           reason_signs_lights_pct=("reason", lambda x: x.isin(["Street Lights", "Signs & Signals"]).mean()),
       )
       .reset_index())

agg["needle_share"] = agg["needle_count"] / agg["total_requests"]

print(f"\nAggregated shape: {agg.shape}")
print(f"Unique neighborhoods: {agg['neighborhood'].nunique()}")
print(f"Year range: {agg['year'].min()} – {agg['year'].max()}")
print(f"\nSample rows:")
agg.head(5)

In [ ]:
print(f"Unique neighborhoods: {agg['neighborhood'].nunique()}")
print(f"\nValue counts:\n{agg['neighborhood'].value_counts().to_string()}")
print("\nNeighborhood is used for aggregation and the risk map,")
print("but excluded from features to avoid location bias.")

In [ ]:
agg = agg.sort_values(["neighborhood", "year", "month"]).reset_index(drop=True)

threshold = agg["needle_count"].quantile(0.75)
agg["high_risk"] = (agg["needle_count"] >= threshold).astype(int)

FEATURE_COLS = [
    "month",
    "total_requests",
    "median_response_hours_log",
    "pct_fast_close",
    "source_app_pct",
    "source_call_pct",
    "source_self_pct",
    "subject_pwd_pct",
    "subject_bwsc_pct",
    "subject_inspections_pct",
    "reason_street_cleaning_pct",
    "reason_enforcement_pct",
    "reason_sanitation_pct",
    "reason_housing_pct",
    "reason_signs_lights_pct",
]
TARGET_COL = "high_risk"

model_df = agg[FEATURE_COLS + [TARGET_COL, "needle_count", "year", "neighborhood"]].dropna().copy()

print(f"High-risk threshold (75th percentile): needle_count >= {threshold:.0f}")
print(f"Class distribution:\n{model_df[TARGET_COL].value_counts().rename({0: 'Low risk', 1: 'High risk'}).to_string()}")
print(f"\nFinal modeling table shape: {model_df.shape}")
print(f"Rows dropped: {len(agg) - len(model_df)}")
print(f"\nFeature columns ({len(FEATURE_COLS)}):")
for c in FEATURE_COLS:
    print(f"  {c}")
print(f"\nTarget: {TARGET_COL}")
print(f"\nDescriptive stats:")
model_df.describe().round(3)

In [ ]:
corr_cols = FEATURE_COLS + [TARGET_COL]
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

corr = model_df[corr_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title("Correlation Matrix — Feature Set + Target")

means = model_df.groupby(TARGET_COL)[FEATURE_COLS].mean().T
means.columns = ["Low risk (0)", "High risk (1)"]
means["Difference"] = means["High risk (1)"] - means["Low risk (0)"]
means_sorted = means.sort_values("Difference")

means_sorted["Difference"].plot.barh(ax=axes[1], color=["C0" if v < 0 else "C3" for v in means_sorted["Difference"]],
                                      edgecolor="black", alpha=0.7)
axes[1].set_title("Feature Mean Difference (High Risk − Low Risk)")
axes[1].set_xlabel("Difference in mean value")
axes[1].axvline(0, color="black", lw=0.8)

plt.tight_layout()
plt.show()

print("Point-biserial correlation with high_risk:")
print(corr[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False).to_string())
print("\nFeature means by class:")
print(means.sort_values("Difference", ascending=False).round(4).to_string())

## Random Forest Classification

We use a **time-based train/test split** — training on earlier years and testing on the most recent year — to simulate how the model would perform in a real deployment where policy makers need *forward-looking* predictions of which neighborhoods will be high-risk.

This avoids temporal leakage (the model never sees future data during training) and evaluates on the period most relevant for current resource allocation decisions.

**Maximizing recall** is the priority for this use case: missing a high-risk neighborhood (false negative) means under-allocating cleanup and outreach resources — a more dangerous failure than over-deploying to a neighborhood that turns out to be low-risk (false positive). To achieve this we:
1. **Increase class weight** for the high-risk class (`{0: 1, 1: 4}`), penalizing missed positives more heavily during training.
2. **Remove the depth cap** (`max_depth=None`) to let the trees capture finer-grained splits that distinguish borderline high-risk cases.
3. **Lower the classification threshold** from the default 0.5 by sweeping thresholds on the training set and selecting the one that achieves at least 95% recall.

In [ ]:
SPLIT_YEAR = 2024

train = model_df[model_df["year"] < SPLIT_YEAR]
test = model_df[model_df["year"] >= SPLIT_YEAR]

X_train = train[FEATURE_COLS]
y_train = train[TARGET_COL]
X_test = test[FEATURE_COLS]
y_test = test[TARGET_COL]

print(f"Train: {len(train)} rows  (years < {SPLIT_YEAR})")
print(f"Test:  {len(test)} rows  (years >= {SPLIT_YEAR})")
print(f"Train year range: {train['year'].min()} – {train['year'].max()}")
print(f"Test year range:  {test['year'].min()} – {test['year'].max()}")
print(f"\nTrain class balance: {y_train.value_counts().to_dict()}")
print(f"Test class balance:  {y_test.value_counts().to_dict()}")

clf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    min_samples_split=4,
    class_weight={0: 1, 1: 4},
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

y_prob_train = clf.predict_proba(X_train)[:, 1]
y_prob_test = clf.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.30, 0.60, 0.01)
results = []
for t in thresholds:
    preds = (y_prob_train >= t).astype(int)
    r = recall_score(y_train, preds)
    p = precision_score(y_train, preds, zero_division=0)
    f = f1_score(y_train, preds, zero_division=0)
    results.append({"threshold": t, "recall": r, "precision": p, "f1": f})

thresh_df = pd.DataFrame(results)
best_row = thresh_df.loc[thresh_df["recall"].ge(0.95).idxmax()]
BEST_THRESHOLD = best_row["threshold"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df["threshold"], thresh_df["recall"], label="Recall", lw=2)
ax.plot(thresh_df["threshold"], thresh_df["precision"], label="Precision", lw=2)
ax.plot(thresh_df["threshold"], thresh_df["f1"], label="F1", lw=2, linestyle="--")
ax.axvline(BEST_THRESHOLD, color="red", linestyle=":", lw=1.5, label=f"Selected threshold = {BEST_THRESHOLD:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Tuning on Training Set (optimizing for recall >= 0.95)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

y_pred_train = (y_prob_train >= BEST_THRESHOLD).astype(int)
y_pred_test = (y_prob_test >= BEST_THRESHOLD).astype(int)

print(f"Model fitted: {clf.n_estimators} trees, max_depth={clf.max_depth}")
print(f"class_weight = {clf.class_weight}")
print(f"\nOptimal threshold (recall >= 0.95 on train): {BEST_THRESHOLD:.2f}")
print(f"At this threshold on train — Recall: {best_row['recall']:.3f}, Precision: {best_row['precision']:.3f}, F1: {best_row['f1']:.3f}")

In [ ]:
metrics = {}
for label, yt, yp in [("Train", y_train, y_pred_train), ("Test", y_test, y_pred_test)]:
    metrics[label] = {
        "Accuracy":  accuracy_score(yt, yp),
        "Precision": precision_score(yt, yp, zero_division=0),
        "Recall":    recall_score(yt, yp, zero_division=0),
        "F1":        f1_score(yt, yp, zero_division=0),
    }

metrics_df = pd.DataFrame(metrics).T
print(f"Classification Metrics (threshold = {BEST_THRESHOLD:.2f})")
print(metrics_df.round(3).to_string())

roc_auc = roc_auc_score(y_test, y_prob_test)
print(f"\nTest ROC-AUC: {roc_auc:.3f}")

print(f"\nClassification Report — Test Set (threshold = {BEST_THRESHOLD:.2f}):")
print(classification_report(y_test, y_pred_test, target_names=["Low risk", "High risk"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Low risk", "High risk"],
            yticklabels=["Low risk", "High risk"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix — Test Set (threshold = {BEST_THRESHOLD:.2f})")

fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
idx = np.argmin(np.abs(roc_thresholds - BEST_THRESHOLD))
axes[1].scatter(fpr[idx], tpr[idx], color="red", s=100, zorder=5,
                label=f"Operating point (t={BEST_THRESHOLD:.2f})")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Test Set")
axes[1].legend(loc="lower right")

plt.suptitle("Random Forest Classification — Performance", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
importances.plot.barh(ax=ax, edgecolor="black", alpha=0.7, color="C2")
ax.set_title("Feature Importance — Random Forest Classifier")
ax.set_xlabel("Importance (mean decrease in impurity)")
plt.tight_layout()
plt.show()

print("Feature importances (descending):")
print(importances.sort_values(ascending=False).to_string())

### Predicted Needle Risk — Boston Neighborhood Map

Using the classifier's **predicted probability of high-risk** on the test set, we compute the mean high-risk probability per neighborhood and plot each neighborhood's centroid on a spatial heatmap of Boston. Warmer/larger dots indicate neighborhoods the model identifies as consistently high-risk — directly telling policy makers where to prioritize cleanup crews and outreach.

In [ ]:
centroids = (df.groupby("neighborhood")
             .agg(lat=("latitude", "mean"), lon=("longitude", "mean"))
             .reset_index())

test_with_nbhd = test.copy()
test_with_nbhd["predicted_class"] = y_pred_test
test_with_nbhd["risk_prob"] = y_prob_test

risk = (test_with_nbhd.groupby("neighborhood")
        .agg(mean_risk_prob=("risk_prob", "mean"),
             pct_flagged=("predicted_class", "mean"),
             actual_high_risk_pct=("high_risk", "mean"),
             mean_needle_count=("needle_count", "mean"))
        .reset_index())

risk = risk.merge(centroids, on="neighborhood", how="left")

def lonlat_to_webmercator(lon, lat):
    x = lon * 20037508.34 / 180
    y = np.log(np.tan((90 + lat) * np.pi / 360)) / (np.pi / 180)
    y = y * 20037508.34 / 180
    return x, y

risk["x"], risk["y"] = lonlat_to_webmercator(risk["lon"].values, risk["lat"].values)

fig, ax = plt.subplots(figsize=(12, 12))

scatter = ax.scatter(
    risk["x"], risk["y"],
    c=risk["mean_risk_prob"],
    s=risk["mean_risk_prob"] * 300 + 40,
    cmap="YlOrRd",
    edgecolors="black",
    linewidths=0.6,
    alpha=0.85,
    zorder=5,
    vmin=0, vmax=1,
)

for _, row in risk.iterrows():
    ax.annotate(
        row["neighborhood"],
        (row["x"], row["y"]),
        fontsize=7,
        fontweight="bold",
        ha="center",
        va="bottom",
        xytext=(0, 8),
        textcoords="offset points",
        zorder=6,
        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7),
    )

PAD = 1500
ax.set_xlim(risk["x"].min() - PAD, risk["x"].max() + PAD)
ax.set_ylim(risk["y"].min() - PAD, risk["y"].max() + PAD)

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, zoom=13)

cbar = plt.colorbar(scatter, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label("Mean Predicted High-Risk Probability", fontsize=11)

ax.set_title("Predicted Needle Pickup Risk by Neighborhood — Test Period", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

print("Neighborhood risk ranking (test period, descending):")
print(risk[["neighborhood", "mean_risk_prob", "pct_flagged", "actual_high_risk_pct", "mean_needle_count"]]
      .sort_values("mean_risk_prob", ascending=False)
      .round(3)
      .to_string(index=False))

### Model Results — Interpretation

**Why maximize recall?** In a public health resource allocation context, the cost of a **false negative** (missing a high-risk neighborhood) far exceeds the cost of a **false positive** (deploying to a neighborhood that turns out to be low-risk). Missing a hotspot means needles remain on streets, creating health hazards. Over-deploying is a minor efficiency cost.

**Metrics breakdown:**
- **Recall** is the primary metric. It answers: "Of the neighborhood-months that truly were high-risk, what proportion did we catch?" By tuning the classification threshold and increasing the class weight for the positive class, we prioritize catching as many true high-risk periods as possible.
- **Precision** answers: "Of the neighborhood-months we flagged as high-risk, what proportion actually were?" With a lowered threshold, precision decreases — we accept more false alarms — but the trade-off is justified for public safety.
- **F1 Score** balances both, but in this application recall matters more than the balanced F1 would suggest.
- **ROC-AUC** measures the model's ranking ability across all thresholds. A value near 1.0 confirms the model has strong discriminative signal regardless of where the threshold is set.

**Threshold tuning:** Rather than using the default 0.5 cutoff, we swept thresholds on the training set and selected the lowest threshold that achieves at least 95% recall. This ensures the model errs on the side of caution — flagging borderline cases rather than missing them.

**Train vs Test gap:** Some gap is expected since the model is trained on 2018–2022 and tested on 2024–2025, and needle patterns may have shifted. The key question is whether test recall remains high — indicating the model generalizes its recall-oriented behavior to unseen future data.

**Feature importance:** Shows which features the Random Forest relies on most for its splits. With no neighborhood identity, lag, or rolling-average features, the model must learn entirely from observable conditions — `month`, `total_requests`, response-time characteristics, and the `source`/`subject`/`reason` proportions — producing actionable insights about what conditions distinguish high-risk from low-risk neighborhood-months, independent of location.

**Confusion matrix:** The four quadrants show: true negatives (correctly identified low-risk), false positives (falsely flagged as high-risk), false negatives (missed high-risk — the most dangerous for policy), and true positives (correctly caught high-risk).

**Risk map:** The spatial heatmap translates the classifier's predicted probabilities into the visual language policy makers use — neighborhoods with larger, redder dots have a higher predicted probability of being high-risk and need more cleanup crews, outreach workers, and disposal infrastructure. This directly answers the research question: *Can high-risk locations be predicted in advance to support proactive resource allocation?*